# 1 — Data Understanding

## What this case study is about

In the US, hospitals don't just get paid what they ask for. They submit a **bill** (called a submitted charge), but Medicare only pays a fraction of that — typically 20–30 cents on the dollar. The gap between what's billed and what's paid varies enormously across hospitals, regions, and procedure types.

This project investigates that gap using three CMS datasets.

---

## The three tables and how they connect

```
T1: Inpatient Payments  (one row = one hospital doing one type of procedure)
        |                        |
        | hospital ID            | procedure code
        ▼                        ▼
T3: Provider Info          T2: DRG Details
(what kind of hospital)    (what kind of procedure)
```

T1 is the **fact table** — it holds all the money data. T2 and T3 are **lookup tables** that add context.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('..') / 'data'
inpatient = pd.read_csv(DATA_DIR / 'Inpatient Payments.CSV')
provider  = pd.read_csv(DATA_DIR / 'Provider Info.csv')
drg       = pd.read_excel(DATA_DIR / 'DRG Details.xls')

print(f'T1 Inpatient Payments: {inpatient.shape[0]:,} rows × {inpatient.shape[1]} columns')
print(f'T2 DRG Details:        {drg.shape[0]:,} rows × {drg.shape[1]} columns')
print(f'T3 Provider Info:      {provider.shape[0]:,} rows × {provider.shape[1]} columns')

T1 Inpatient Payments: 196,086 rows × 15 columns
T2 DRG Details:        543 rows × 8 columns
T3 Provider Info:      5,426 rows × 38 columns


---

## T1 — Inpatient Payments

**What it is**: The main table. Every row represents one **hospital** performing one **type of procedure** on some number of patients.

**Join key**: `Rndrng_Prvdr_CCN` → links to T3's `Facility ID` (same 6-digit hospital code)

| Column | Plain English |
|--------|---------------|
| `Rndrng_Prvdr_CCN` | The hospital's unique 6-digit ID number assigned by Medicare |
| `Rndrng_Prvdr_Org_Name` | The hospital's name |
| `Rndrng_Prvdr_State_Abrvtn` | 2-letter US state abbreviation (used for geographic analysis) |
| `DRG_Cd` | A code for the type of procedure/treatment. Links to T2. |
| `DRG_Desc` | The plain-language name of that procedure |
| `Tot_Dschrgs` | How many patients were discharged for this hospital + procedure combination |
| `Avg_Submtd_Cvrd_Chrg` | **The sticker price** — what the hospital billed Medicare on average |
| `Avg_Mdcr_Pymt_Amt` | **What Medicare actually paid** on average |
| `Avg_Tot_Pymt_Amt` | Total actual payment (Medicare + other adjustments) |

> **Example row in plain English**: *"Hospital #050441 in California performed knee replacements (DRG 470) on 47 patients. They billed \$65,000 on average. Medicare paid \$15,000 on average."*

In [2]:
cols_to_show = [
    'Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_State_Abrvtn',
    'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs',
    'Avg_Submtd_Cvrd_Chrg', 'Avg_Mdcr_Pymt_Amt'
]
inpatient[cols_to_show].head(4)

,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_State_Abrvtn,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Mdcr_Pymt_Amt
0,10001,Southeast Alabama Medical Center,AL,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,32,126083.187500,25320.750000
1,10001,Southeast Alabama Medical Center,AL,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,31,115948.419350,20814.032258
2,10001,Southeast Alabama Medical Center,AL,37,EXTRACRANIAL PROCEDURES WITH MCC,13,78837.307692,14301.230769
3,10001,Southeast Alabama Medical Center,AL,38,EXTRACRANIAL PROCEDURES WITH CC,17,56492.823529,6767.470588


---

## T2 — DRG Details

**What it is**: A lookup table for each type of procedure. DRG = *Diagnosis Related Group* — CMS groups all hospital procedures into ~750 categories.

**Join key**: `DRGV22` → links to T1's `DRG_Cd`

| Column | Plain English |
|--------|---------------|
| `DRGV22` | The procedure code (the join key) |
| `MDC` | **Major Diagnostic Category** — a broader grouping of procedures into 25 body-system buckets. e.g. MDC 05 = Heart, MDC 08 = Bones & Joints |
| `DRG TITLE` | The name of the procedure |
| `RELATIVE WEIGHTS` | **Complexity score** — how resource-intensive this procedure is. A heart transplant (weight ≈ 25) is far more complex than an appendectomy (weight ≈ 1). Medicare uses this to set base reimbursement. |
| `GEOMETRIC MEAN LOS` | Typical hospital stay length in days (geometric mean is used because LOS data is skewed — a few very long stays inflate the arithmetic average) |

> **Example**: DRG 470 (knee replacement) has relative weight ≈ 2.1 and a typical stay of 2 days. A heart transplant (DRG 1) has relative weight ≈ 25 and typical stay of 14 days.

In [3]:
drg[['DRGV22', 'MDC', 'DRG TITLE', 'RELATIVE WEIGHTS', 'GEOMETRIC MEAN LOS']].head(6)

,DRGV22,MDC,DRG TITLE,RELATIVE WEIGHTS,GEOMETRIC MEAN LOS
0,1,01,CRANIOTOMY AGE >17 W CC,3.3344,7.5
1,2,01,CRANIOTOMY AGE >17 W/O CC,1.9467,3.6
2,3,01,CRANIOTOMY AGE 0-17,1.9767,12.7
3,4,01,NO LONGER VALID,0.0000,0.0
4,5,01,NO LONGER VALID,0.0000,0.0
5,6,01,CARPAL TUNNEL RELEASE,0.7850,2.2


---

## T3 — Provider Info

**What it is**: A profile of each hospital — what type it is and who owns it.

**Join key**: `Facility ID` → links to T1's `Rndrng_Prvdr_CCN`

| Column | Plain English |
|--------|---------------|
| `Facility ID` | The 6-digit hospital ID (same as CCN in T1, just named differently) |
| `Facility Name` | Hospital name |
| `Hospital Type` | The category of care the hospital provides. The main types are: **Acute Care** (general full-service hospitals), **Critical Access** (small rural hospitals, ≤25 beds), **Psychiatric**, **Children's** |
| `Hospital Ownership` | Who owns the hospital: **Voluntary non-profit** (no shareholders, reinvests profits), **Proprietary** (for-profit, has shareholders), **Government** (federal, state, or local) |
| `Emergency Services` | Y/N — does the hospital have an emergency department? |

> **Why ownership matters**: Non-profit hospitals and for-profit hospitals have different incentives. For-profits tend to charge more relative to Medicare payment (higher markup). We test this hypothesis in the analysis.

In [4]:
provider[['Facility ID', 'Facility Name', 'Hospital Type', 'Hospital Ownership', 'Emergency Services']].head(5)

,Facility ID,Facility Name,Hospital Type,Hospital Ownership,Emergency Services
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,Acute Care Hospitals,Government - Hospital District or Authority,Yes
1,010005,MARSHALL MEDICAL CENTERS,Acute Care Hospitals,Government - Hospital District or Authority,Yes
2,010006,NORTH ALABAMA MEDICAL CENTER,Acute Care Hospitals,Proprietary,Yes
3,010007,MIZELL MEMORIAL HOSPITAL,Acute Care Hospitals,Voluntary non-profit - Private,Yes
4,010008,CRENSHAW COMMUNITY HOSPITAL,Acute Care Hospitals,Proprietary,Yes


---

## Quick dataset health check

In [5]:
print('T1 — Unique hospitals:', inpatient['Rndrng_Prvdr_CCN'].nunique())
print('T1 — Unique DRGs:',      inpatient['DRG_Cd'].nunique())
print('T1 — States covered:',   inpatient['Rndrng_Prvdr_State_Abrvtn'].nunique())
print('T1 — Total discharges:', f"{inpatient['Tot_Dschrgs'].sum():,.0f}")
print()
print('T2 — DRG codes:', len(drg))
print('T2 — MDC categories:', drg['MDC'].nunique())
print()
print('T3 — Hospital ownership types:', provider['Hospital Ownership'].nunique())
print('T3 — Hospital types:', provider['Hospital Type'].nunique())

T1 — Unique hospitals: 3182
T1 — Unique DRGs: 563
T1 — States covered: 51
T1 — Total discharges: 7,398,125

T2 — DRG codes: 543
T2 — MDC categories: 26

T3 — Hospital ownership types: 12
T3 — Hospital types: 8


---

## Join key validation

The PDF warns that T1 calls the hospital ID `provider_id` and T3 calls it `Facility ID` — but they're the same value. The only catch is leading zeros.
We validate the match rate before doing any analysis.

In [6]:
# Normalise both sides: strip leading zeros
t1_ids   = set(inpatient['Rndrng_Prvdr_CCN'].astype(str).str.lstrip('0'))
t3_ids   = set(provider['Facility ID'].astype(str).str.lstrip('0'))
t1_in_t3 = len(t1_ids & t3_ids) / len(t1_ids)
print(f'T1 hospital IDs found in T3: {t1_in_t3:.1%}')
print(f'T2 DRG codes: {len(drg)} total, {len(set(drg["DRGV22"]) & set(inpatient["DRG_Cd"].dropna().astype(int)))} match T1')

T1 hospital IDs found in T3: 90.4%
T2 DRG codes: 543 total, 344 match T1
